# VisionX AI – AI-Based Multi-Disease Eye Disease Detection System
## Notebook 02: Pre-trained Model Fine-Tuning and Performance Evaluation

**VisionX AI** assists ophthalmologists in automatically detecting seven eye conditions using retinal fundus images. In the previous phase, we split our data and saved the preprocessed split summary. Now, we will load this split dataset, construct a data pipeline, configure a pre-trained network (ResNet18 / MobileNetV3), train the classification head on the CPU, and evaluate the model's performance on the test set.

### 1. Imports and Workspace Configuration
We import standard scientific libraries and our custom, modular training utilities (`RetinalDataset`, `build_model`, `train_model`, `evaluate_model`).

In [ ]:
import os
import sys
import torch
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure the workspace root is in python path to load local modules
sys.path.append(os.path.abspath(".."))

from dataset import RetinalDataset, CLASSES
from model import build_model
from train import train_model
from evaluate import evaluate_model

### 2. Configure Training Hyperparameters
Configure key hyperparameters. Since training is performed on the CPU, we default the image resolution to `128x128` and set a moderate epoch count to allow training to complete in a reasonable timeframe.

In [ ]:
DATA_DIR = "../PreparedData"
IMAGE_SIZE = 128
BATCH_SIZE = 32
EPOCHS = 10
LEARNING_RATE = 0.001
MODEL_NAME = "tv_resnet18"
MODEL_PATH = "../best_model.pth"
OUTPUT_DIR = "../evaluation_results"
PATIENCE = 3

### 3. Run the Training Pipeline
We execute the modular training workflow. This calculates inverse class frequencies for weighting the cross-entropy loss, builds the ResNet18 model with frozen backbone parameters, trains the custom linear classification head, and runs early stopping based on validation Macro F1-score.

In [ ]:
train_model(
    data_dir=DATA_DIR,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    lr=LEARNING_RATE,
    model_name=MODEL_NAME,
    model_path=MODEL_PATH,
    dry_run=False,
    patience=PATIENCE
)

### 4. Evaluate the Model on the Test Set
Run inference on the test split using the saved best weights, and output the per-class metrics and confusion matrix.

In [ ]:
evaluate_model(
    data_dir=DATA_DIR,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    model_name=MODEL_NAME,
    model_path=MODEL_PATH,
    output_dir=OUTPUT_DIR
)

### 5. Visualize the Confusion Matrix
We display the confusion matrix saved during evaluation to inspect which classes are well-predicted or commonly confused.

In [ ]:
from IPython.display import Image, display
cm_image_path = os.path.join(OUTPUT_DIR, "confusion_matrix.png")

if os.path.exists(cm_image_path):
    display(Image(filename=cm_image_path))
else:
    print("Confusion matrix image not found. Ensure evaluation script was run.")

### 6. Conclusions and Recommendations
- **Macro F1 vs Accuracy**: Compare class performance. Note how class weights improved performance on rarer classes (like Pathological Myopia) compared to a naive model.
- **CPU Limitations**: For production-level accuracy, training should be scaled up to a GPU cluster using a larger resolution ($224 \times 224$ or $384 \times 384$) and fully fine-tuning the model backbone (unfreezing weights).